# SIV-WAM 手撕学习 · 04：Token Layout 与联合序列

本节把第三部分得到的三个训练流转换为 Transformer token：

```text
latent_dict
mask_dict
 action_dict
    ↓
video / mask patchify
 action flatten
    ↓
六类 stream embedding
    ↓
[vid_noisy | vid_cond | mask_noisy | mask_cond | act_noisy | act_cond]
    ↓
联合序列 [1, B×L, D]
```

本节只手写 token layout，不加载 Wan 权重。重点是理解 token 数、拼接顺序、grid id 和 split boundary。

## 本节必须掌握的核心结论

当前 fake batch：

```text
video  [B,4,3,8,10]
mask   [B,4,3,4,10]
action [B,30,3,16,1]
patch_size = (1,2,2)
```

因此：

```text
video token 数  = 3×(8/2)×(10/2) = 60
mask token 数   = 3×(4/2)×(10/2) = 30
action token 数 = 3×16×1 = 48
```

六类 stream 的每样本总 token 数：

```text
60 + 60 + 30 + 30 + 48 + 48 = 276
```

In [14]:
from pathlib import Path

import torch
from torch import nn

PROJECT_ROOT = Path.cwd()
B, C, F_LEN = 2, 4, 3
VIDEO_H, VIDEO_W = 8, 10
MASK_H, MASK_W = 4, 10
ACTION_DIM, ACTION_N = 30, 16
PATCH_SIZE = (1, 2, 2)
HIDDEN_DIM = 32

video = torch.randn(B, C, F_LEN, VIDEO_H, VIDEO_W)
mask = torch.randn(B, C, F_LEN, MASK_H, MASK_W)
action = torch.randn(B, ACTION_DIM, F_LEN, ACTION_N, 1)
video_timesteps = torch.tensor([[10., 30., 50.], [10., 30., 50.]])
mask_timesteps = video_timesteps.clone()
action_timesteps = torch.tensor([[15., 35., 55.], [15., 35., 55.]])

print('video:', video.shape)
print('mask:', mask.shape)
print('action:', action.shape)

video: torch.Size([2, 4, 3, 8, 10])
mask: torch.Size([2, 4, 3, 4, 10])
action: torch.Size([2, 30, 3, 16, 1])


## 1. 手写 latent patchify

生产代码 `_input_embed()` 中的关键变换是：

```python
'b c (f p1) (h p2) (w p3) -> b (f h w) (c p1 p2 p3)'
```

本节先不做 embedding，只完成 patch 展平。

目标：

```text
video [B,C,F,H,W]
    → [B,F×H/p2×W/p3,C×p1×p2×p3]
```

对于当前 video：

```text
[B,4,3,8,10] → [B,60,16]
```

In [25]:
def patchify_latent_todo(latents, patch_size):
    # TODO 1：检查 F/H/W 能否被 p1/p2/p3 整除
    B,C,F,H,W = latents.shape
    p1,p2,p3 = patch_size
    if F % patch_size[0] != 0 or H % patch_size[1] != 0 or W % patch_size[2] != 0:
        raise ValueError(f"F,H,W = {F},{H},{W}, patch_size = {patch_size}")
    # TODO 2：将每个 p1×p2×p3 patch 展平为 token feature
    x = latents.reshape(B,C,F//p1,p1,H//p2,p2,W//p3,p3).permute(0,2,4,6,1,3,5,7).contiguous()
    tokens = x.reshape(B,F*H*W//p2//p3,C*p1*p2*p3)
    # TODO 3：返回 [B, token_count, C*p1*p2*p3]
    return  tokens


In [26]:
# 完成 patchify_latent_todo 后运行
video_patch = patchify_latent_todo(video, PATCH_SIZE)
mask_patch = patchify_latent_todo(mask, PATCH_SIZE)
print('video patch:', video_patch.shape)
print('mask patch:', mask_patch.shape)
assert video_patch.shape == (B, 60, 16)
assert mask_patch.shape == (B, 30, 16)

video patch: torch.Size([2, 60, 16])
mask patch: torch.Size([2, 30, 16])


## 2. 手写 action tokenization

action 不是图像，不使用 `(1,2,2)` patch。

生产代码使用：

```python
'b c f h w -> b (f h w) c'
```

当前 action：

```text
[B,30,3,16,1] → [B,48,30]
```

In [27]:
def tokenize_action_todo(actions):
    # TODO：将 [B,C,F,N,1] 转成 [B,F*N,C]
    b,c,f,n,_ = action.shape
    x = action.reshape(b,f*n*1,c)
    return x

In [28]:
action_tokens = tokenize_action_todo(action)
print('action tokens:', action_tokens.shape)
assert action_tokens.shape == (B, 48, 30)

action tokens: torch.Size([2, 48, 30])


## 3. 手写 video/mask/action embedding

生产模型会使用：

```text
video/mask patch feature → patch_embedding_mlp
action feature            → action_embedder
```

这里用简单的 `nn.Linear` 代替 Wan 的 embedding，只验证维度。

In [29]:
video_patch_embedder = nn.Linear(C * PATCH_SIZE[0] * PATCH_SIZE[1] * PATCH_SIZE[2], HIDDEN_DIM)
mask_patch_embedder = nn.Linear(C * PATCH_SIZE[0] * PATCH_SIZE[1] * PATCH_SIZE[2], HIDDEN_DIM)
action_embedder = nn.Linear(ACTION_DIM, HIDDEN_DIM)

video_hs = video_patch_embedder(video_patch)
mask_hs = mask_patch_embedder(mask_patch)
action_hs = action_embedder(action_tokens)

print(video_hs.shape, mask_hs.shape, action_hs.shape)
assert video_hs.shape == (B, 60, HIDDEN_DIM)
assert mask_hs.shape == (B, 30, HIDDEN_DIM)
assert action_hs.shape == (B, 48, HIDDEN_DIM)

torch.Size([2, 60, 32]) torch.Size([2, 30, 32]) torch.Size([2, 48, 32])


## 4. 构造六类 stream

六类 stream 的顺序必须严格保持：

```text
1. vid_noisy
2. vid_cond
3. mask_noisy
4. mask_cond
5. act_noisy
6. act_cond
```

本节先用同一个 hidden tensor 代替 noisy/condition，重点只验证布局。

In [ ]:
def build_six_streams_todo(video_hs, mask_hs, action_hs):
    # TODO 1：复制出 noisy/condition 两份
    # TODO 2：每个 stream 从 [B,L,D] 变成 [1,B*L,D]
    # TODO 3：按照生产代码顺序返回 list
    raise NotImplementedError('请先完成 build_six_streams_todo')

In [ ]:
streams = build_six_streams_todo(video_hs, mask_hs, action_hs)
print([tuple(x.shape) for x in streams])
assert len(streams) == 6
assert [x.shape[1] for x in streams] == [B*60, B*60, B*30, B*30, B*48, B*48]
full_hidden = torch.cat(streams, dim=1)
print('full hidden:', full_hidden.shape)
assert full_hidden.shape == (1, B * 276, HIDDEN_DIM)

## 5. 手写 grid id

生产代码使用 `get_mesh_id()` 构造：

```text
[f_position, h_position, w_position, stream_type]
```

视频和 mask 的空间网格不同，因此必须分别生成：

```text
video grid：F×(H/2)×(W/2) = 60
mask grid： F×(H/2)×(W/2) = 30
action grid：F×N×1 = 48
```

当前先完成一个简单版本：返回 `[4,L]` 的 grid id。最后一行可以用来标记 stream 类型。

In [ ]:
def build_grid_ids_todo(F_len, H, W, patch_size, action=False, stream_type=0):
    # TODO 1：计算 patch 后的 f/h/w 网格
    # TODO 2：用 torch.meshgrid 生成坐标
    # TODO 3：action=True 时，将 h/w 解释为 action position，而非图像空间
    # TODO 4：拼接 stream_type 作为第 4 行
    # TODO：返回 [4,L]
    raise NotImplementedError('请先完成 build_grid_ids_todo')

In [ ]:
video_grid = build_grid_ids_todo(F_LEN, VIDEO_H, VIDEO_W, PATCH_SIZE, action=False, stream_type=0)
mask_grid = build_grid_ids_todo(F_LEN, MASK_H, MASK_W, PATCH_SIZE, action=False, stream_type=1)
action_grid = build_grid_ids_todo(F_LEN, ACTION_N, 1, (1, 1, 1), action=True, stream_type=2)

print('video grid:', video_grid.shape)
print('mask grid:', mask_grid.shape)
print('action grid:', action_grid.shape)
assert video_grid.shape == (4, 60)
assert mask_grid.shape == (4, 30)
assert action_grid.shape == (4, 48)

## 6. split boundary

Transformer 输出后需要把联合序列拆回不同 stream。

当前每个样本的 split size 是：

```python
[60, 60, 30, 30, 48, 48]
```

请手写一个函数验证：

```text
cat(streams) → split(full_hidden) → 原始 streams
```

In [ ]:
def split_streams_todo(full_hidden, split_sizes):
    # TODO：沿 token 维 dim=1 拆分
    raise NotImplementedError('请先完成 split_streams_todo')

split_sizes = [60, 60, 30, 30, 48, 48]
recovered = split_streams_todo(full_hidden, split_sizes)
assert len(recovered) == 6
for original, restored in zip(streams, recovered):
    assert torch.equal(original, restored)

## 本节验收标准

你需要通过以下检查：

```text
video patch： [B,60,16]
mask patch：  [B,30,16]
action token：[B,48,30]
每个 stream：[1,B*L,32]
联合序列：  [1,B*276,32]
grid id：    video 60、mask 30、action 48
split 后能恢复六个 stream
```

完成后你应该能解释：

1. 为什么 video 和 mask 的 token 数不同；
2. 为什么 action 不使用图像 patch；
3. 为什么联合序列必须保持六类 stream 的固定顺序；
4. 为什么 Transformer 输出后还需要 split；
5. 为什么 mask 必须有自己的 grid id。

下一节是 `05_attention_mask.ipynb`，将手写 `seq_ids`、`frame_ids`、`noise_ids` 及其 attention 规则。